# Прогноз последним значением

\(\widehat x_{i,t+1}=x_{i,t}\). Обучаемых весов нет; заполнение входов и маски общие со всеми моделями.

Каждый запуск этого ноутбука обрабатывает **все три набора**: синтетику, MOEX и METR-LA.
Наборы загружаются по очереди. Вспомогательные функции находятся в `src/forecast_*.py`.
Результаты каждого опыта и сводный CSV сохраняются автоматически.

Нужны `numpy`, `pandas`, `torch`. Отдельные графовые библиотеки не требуются.

Persistence не обучается. PyTorch используется общим загрузчиком и оценкой; настройки эпох и оптимизатора на этот прогноз не влияют.


In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import torch
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
PROJECT_ROOT = PROJECT_ROOT.resolve()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from forecast_batch import run_batch
from forecast_experiment import collect_results, fixed_graph
from forecast_models import build_adjacency

print("Проект:", PROJECT_ROOT)
print("PyTorch:", torch.__version__)


Проект: /home/loktar/Desktop/money_coursework
PyTorch: 2.15.0+cpu


## Настройки серии

Каждый ноутбук запускает свою модель на `synthetic`, `moex`, `metr_la` по очереди.
История и горизонт читаются из `schema.json`; текущая постановка — 60 наблюдений и один шаг вперёд.
Используйте одинаковые `DATASETS`, `CONFIG`, `SEEDS`, `RUN_TAG` во всех четырёх ноутбуках.
Один seed — пилот; для оценки разброса позже задайте `[0, 1, 2]` во всех четырёх.

`EVALUATE_TEST=True`: после выбора лучшей эпохи по val рассчитывается test.
Test не участвует в обучении и выборе эпохи. Для новой настройки по val можно задать `False`.
Уже просмотренный test учитываем как исследовательскую оценку при дальнейшей настройке.


In [2]:
MODEL_KIND = "persistence"
DATASETS = ["synthetic", "moex", "metr_la"]
SEEDS = [0]
RUN_TAG = "pilot_v1"
EVALUATE_TEST = True

CONFIG = {
    "hidden_size": 16,
    "batch_size": 32,
    "max_epochs": 15,
    "patience": 4,
    "learning_rate": 0.001,
    "weight_decay": 0.0001,
    "gradient_clip": 1.0,
    "top_k": 3,
    "cpu_threads": 2,
    "device": "cpu",
    "min_delta": 0.0,
}

DATA_ROOT = PROJECT_ROOT / "data/processed"
RESULTS_ROOT = PROJECT_ROOT / "results"
print("Модель:", MODEL_KIND)
print("Наборы:", DATASETS)


Модель: persistence
Наборы: ['synthetic', 'moex', 'metr_la']


## Общий загрузчик и покрытие целей

CSV каждого набора читаются один раз при его запуске. Масштабирование каждого ряда обучается только по наблюдаемым train-значениям.
Пропуски входа заполняются прошлым значением только внутри выбранного окна, затем обучающим средним.
Цели остаются пропущенными; `y_mask` применяется и в функции потерь, и во всех метриках.

Таблица ниже показывает **фактическое число оцениваемых целей** каждой вершины.
Вершина с нулевым покрытием остаётся в составе, но не получает выдуманную нулевую ошибку.
Состав акций по доступности test не пересматривается.

Функция ниже вызывается автоматически внутри цикла по наборам.


In [3]:
def inspect_dataset(data, dataset_name):
    # Keep inspection local so the previous dataset can be released.
    display(pd.DataFrame([
        {"dataset": dataset_name, "split": split, "samples": len(data.dataset(split)),
         "nodes": len(data.nodes), "history": data.history, "features_per_node": 1}
        for split in ("train", "val", "test")
    ]))
    coverage = data.quality().pivot(index="node", columns="split", values="evaluated_targets")
    display(coverage.head(12))
    zero_coverage = coverage.loc[coverage.eq(0).any(axis=1)]
    if len(zero_coverage):
        print("Вершины без оцениваемых целей хотя бы в одном разбиении:")
        display(zero_coverage)
    example = data.dataset("val")[0]
    print("Один вход:", example["X"].shape, "цель:", example["y_norm"].shape)


## Запуск на трёх наборах

Для каждого входного окна прогноз равен последнему доступному значению каждой вершины.
Обучения и выбора эпохи нет. На MOEX прогноз log(RV) переводится в RV через exp.
MAE/RMSE считаются в исходной шкале; QLIKE используется только для положительного прогноза RV на MOEX.

После каждого завершённого опыта обновляется `summary.csv` этой серии.
В `status.csv` виден результат каждого набора и seed; `errors.csv` содержит ошибки.
При ошибке следующий набор всё равно запускается; в конце частичная серия явно отмечается.
Повторный запуск переиспользует **завершённые** опыты с той же конфигурацией.
Прерванное обучение текущий runner не возобновляет: для него потребуется новый `RUN_TAG`.


In [4]:
batch = run_batch(
    DATA_ROOT, MODEL_KIND, RESULTS_ROOT,
    datasets=DATASETS, seeds=SEEDS, config=CONFIG,
    evaluate_test=EVALUATE_TEST, run_tag=RUN_TAG,
    inspect_dataset=inspect_dataset,
)
reports = batch["reports"]
summary = batch["summary"]
SUMMARY_DIR = batch["summary_dir"]
display(batch["status"])
display(summary)
print("Сводные результаты:", SUMMARY_DIR / "summary.csv")



=== persistence: synthetic ===


,dataset,split,samples,nodes,history,features_per_node
0,synthetic,train,4140,12,60,1
1,synthetic,val,600,12,60,1
2,synthetic,test,1201,12,60,1


split,test,train,val
node,,,
node_0,1201,4140,600
node_1,1201,4140,600
node_10,1201,4140,600
node_11,1201,4140,600
node_2,1201,4140,600
node_3,1201,4140,600
node_4,1201,4140,600
node_5,1201,4140,600
node_6,1201,4140,600


Один вход: (60, 12, 1) цель: (12,)
Using selected checkpoint for test; training is not repeated
Saved: /home/loktar/Desktop/money_coursework/results/synthetic/persistence/seed_000/7090c1700dcc

=== persistence: moex ===


,dataset,split,samples,nodes,history,features_per_node
0,moex,train,870,10,60,1
1,moex,val,252,10,60,1
2,moex,test,254,10,60,1


split,test,train,val
node,,,
GAZP,254,867,252
GMKN,254,870,247
LKOH,254,870,252
MGNT,254,870,252
NVTK,254,870,252
PLZL,251,870,252
ROSN,254,870,252
SBER,254,870,252
VTBR,254,870,248


... (5 outputs left out)

Один вход: (60, 207, 1) цель: (207,)
Saved: /home/loktar/Desktop/money_coursework/results/metr_la/persistence/seed_000/9ae4bbf41e78
Batch summaries: /home/loktar/Desktop/money_coursework/results/summaries/persistence/20261124T081851_533783Z


,dataset,model,seed,status,error,output_dir
0,synthetic,persistence,0,completed,,/home/loktar/Desktop/money_coursework/results/...
1,moex,persistence,0,completed,,/home/loktar/Desktop/money_coursework/results/...
2,metr_la,persistence,0,completed,,/home/loktar/Desktop/money_coursework/results/...


,dataset,model,seed,split,primary_metric,primary_score,mae,rmse,qlike,normalized_mse,n_observed,n_origins,best_epoch,parameters,fit_seconds,comparison_id,run_id,output_dir
0,synthetic,persistence,0,val,mae,0.096973,0.096973,0.121423,NaN,1.137934,7200,600,None,0,0.0,36989cce73a0,7090c1700dcc,/home/loktar/Desktop/money_coursework/results/...
1,synthetic,persistence,0,test,mae,0.096982,0.096982,0.121153,NaN,1.132791,14412,1201,None,0,0.0,36989cce73a0,7090c1700dcc,/home/loktar/Desktop/money_coursework/results/...
2,moex,persistence,0,val,qlike,-7.368911,0.000167,0.000364,-7.368911,0.637047,2371,252,None,0,0.0,362a12d77352,87b39ade7dac,/home/loktar/Desktop/money_coursework/results/...
3,moex,persistence,0,test,qlike,-7.147614,0.000190,0.000805,-7.147614,0.683824,2283,254,None,0,0.0,362a12d77352,87b39ade7dac,/home/loktar/Desktop/money_coursework/results/...
4,metr_la,persistence,0,val,mae,2.631560,2.631560,4.372454,NaN,0.315757,660000,3246,None,0,0.0,c43da30bee45,9ae4bbf41e78,/home/loktar/Desktop/money_coursework/results/...
5,metr_la,persistence,0,test,mae,2.812904,2.812904,4.620502,NaN,0.337782,1247252,6189,None,0,0.0,c43da30bee45,9ae4bbf41e78,/home/loktar/Desktop/money_coursework/results/...


Сводные результаты: /home/loktar/Desktop/money_coursework/results/summaries/persistence/20261124T081851_533783Z/summary.csv


## Сохранённые результаты и примеры

Подробные результаты: `results/<dataset>/<model>/seed_<seed>/<run_id>/`.
Там находятся `run.json`, метрики, ошибки по датам `*_losses.csv`, метрики по вершинам,
прогнозы с исходной маской в `*_predictions.npz`. У нейросетей также сохраняются история обучения и лучшая модель.

Сводка этой серии: `results/summaries/<model>/<дата_UTC>/summary.csv`.
Одна строка — один набор, seed и разбиение val/test; при одном seed и трёх успешных наборах получится 6 строк.
В сводке есть основная метрика, MAE, RMSE, QLIKE для MOEX, число целей, время обучения и путь к опыту.
Разные серии получают отдельные папки. Ниже показаны примеры для каждого успешно обработанного набора.


In [5]:
preview_split = "test" if EVALUATE_TEST else "val"
for dataset_name in DATASETS:
    dataset_reports = [report for report in reports if report["dataset"] == dataset_name]
    if not dataset_reports:
        continue
    run_dir = Path(dataset_reports[0]["output_dir"])
    print(dataset_name, "—", run_dir)
    display(pd.read_csv(run_dir / f"{preview_split}_per_node.csv").head(12))
    with np.load(run_dir / f"{preview_split}_predictions.npz") as saved:
        preview = pd.DataFrame({
            "node": saved["node_order"], "observed": saved["mask"][0],
            "actual_raw": saved["target_raw"][0], "prediction_raw": saved["prediction_raw"][0],
        })
    display(preview.head(12))


synthetic — /home/loktar/Desktop/money_coursework/results/synthetic/persistence/seed_000/7090c1700dcc


,node_id,node,name,observed_pairs,n_observed,mae,rmse
0,0,node_0,node_0,1201,1201,0.099435,0.124073
1,1,node_1,node_1,1201,1201,0.093084,0.116834
2,2,node_2,node_2,1201,1201,0.098075,0.123196
3,3,node_3,node_3,1201,1201,0.097512,0.120459
4,4,node_4,node_4,1201,1201,0.094566,0.117675
5,5,node_5,node_5,1201,1201,0.096915,0.121319
6,6,node_6,node_6,1201,1201,0.100737,0.126380
7,7,node_7,node_7,1201,1201,0.094794,0.118687
8,8,node_8,node_8,1201,1201,0.095421,0.119392
9,9,node_9,node_9,1201,1201,0.100787,0.125317


,node,observed,actual_raw,prediction_raw
0,node_0,True,-0.018677,-0.008965
1,node_1,True,0.002347,0.021910
2,node_2,True,-0.135561,-0.000086
3,node_3,True,0.063401,0.091658
4,node_4,True,0.048733,-0.035254
5,node_5,True,-0.123078,-0.084055
6,node_6,True,-0.013832,-0.154905
7,node_7,True,-0.040185,0.027415
8,node_8,True,0.174519,0.136875
9,node_9,True,-0.034389,-0.040633


moex — /home/loktar/Desktop/money_coursework/results/moex/persistence/seed_000/87b39ade7dac


,node_id,node,name,observed_pairs,n_observed,mae,rmse,qlike
0,0,SBER,SBER,254,254,0.000090,0.000189,-7.937294
1,1,GAZP,GAZP,254,254,0.000217,0.000393,-7.019056
2,2,LKOH,LKOH,254,254,0.000128,0.000261,-7.436969
3,3,GMKN,GMKN,254,254,0.000153,0.000233,-7.058267
4,4,YNDX,YNDX,0,0,NaN,NaN,NaN
5,5,ROSN,ROSN,254,254,0.000140,0.000226,-7.301601
6,6,PLZL,PLZL,251,251,0.000172,0.000321,-7.058526
7,7,MGNT,MGNT,254,254,0.000173,0.000303,-7.147409
8,8,NVTK,NVTK,254,254,0.000249,0.000452,-6.719431
9,9,VTBR,VTBR,254,254,0.000390,0.002248,-6.648921


,node,observed,actual_raw,prediction_raw
0,SBER,True,0.000196,0.000130
1,GAZP,True,0.000641,0.000330
2,LKOH,True,0.000351,0.000215
3,GMKN,True,0.000510,0.000239
4,YNDX,False,NaN,0.000329
5,ROSN,True,0.000478,0.000180
6,PLZL,True,0.000058,0.000026
7,MGNT,True,0.000848,0.000600
8,NVTK,True,0.000861,0.000293
9,VTBR,True,0.000401,0.000199


metr_la — /home/loktar/Desktop/money_coursework/results/metr_la/persistence/seed_000/9ae4bbf41e78


,node_id,node,name,observed_pairs,n_observed,mae,rmse
0,0,773869,773869,6188,6188,2.233973,4.154989
1,1,767541,767541,6189,6189,1.975631,3.053240
2,2,767542,767542,6189,6189,2.103999,3.627639
3,3,717447,717447,6185,6185,2.864586,4.415684
4,4,717446,717446,5963,5963,3.772245,5.356438
5,5,717445,717445,6185,6185,3.802344,5.688712
6,6,773062,773062,6185,6185,2.798898,4.235849
7,7,767620,767620,6186,6186,2.179553,3.167953
8,8,737529,737529,6088,6088,2.864063,4.476317
9,9,717816,717816,6186,6186,2.713083,4.602394


,node,observed,actual_raw,prediction_raw
0,773869,True,64.750,63.555557
1,767541,True,56.375,61.222221
2,767542,True,69.375,68.000000
3,717447,True,58.000,59.777779
4,717446,True,66.000,63.888889
5,717445,True,44.375,49.888889
6,773062,True,63.125,65.777779
7,767620,True,52.250,60.111111
8,737529,True,65.750,60.555557
9,717816,True,68.000,67.666664


## Сопоставление четырёх запусков

Сравнивайте строки с одним `comparison_id`: он фиксирует данные, нормировку, общие настройки и версии кода.
Метрики разных наборов данных имеют разные единицы и напрямую не сопоставляются.

Обычная LSTM здесь многомерная, а LSTM–GNN использует общий по вершинам временной кодировщик.
Поэтому их сравнение относится к двум семействам моделей, а не к изолированному добавлению одного слоя.
Главная проверка обновления графа — **static против dynamic** при одинаковых seed и настройках.

Текущая синтетика со статическим истинным графом служит контрольным сценарием.
Для проверки пользы изменения топологии позже потребуется синтетика с меняющимися связями.

`comparison.csv` собирает уже завершённые запуски других моделей с теми же протоколом и seed
внутри каждого набора. После запуска всех четырёх ноутбуков таблица последнего включает все четыре модели.
При одном seed и наличии val/test это 24 строки. Ошибки разных наборов не усредняются.


In [6]:
comparison_parts = []
for dataset_name in DATASETS:
    current_ids = summary.loc[summary["dataset"].eq(dataset_name), "comparison_id"].unique()
    for split in (("val", "test") if EVALUATE_TEST else ("val",)):
        table = collect_results(RESULTS_ROOT, dataset_name, split=split, run_tag=RUN_TAG)
        if table.empty:
            continue
        table = table.loc[table["comparison_id"].isin(current_ids) & table["seed"].isin(SEEDS)].copy()
        if not table.empty:
            table.insert(0, "dataset", dataset_name)
            comparison_parts.append(table)
comparison = pd.concat(comparison_parts, ignore_index=True) if comparison_parts else pd.DataFrame(
    columns=["dataset", "model", "seed", "split", "comparison_id", "mae", "rmse", "qlike"]
)
comparison.to_csv(SUMMARY_DIR / "comparison.csv", index=False)
display(comparison)
print("Сопоставимые запуски:", SUMMARY_DIR / "comparison.csv")

failed = batch["status"].loc[batch["status"]["status"].ne("completed")]
if not failed.empty:
    display(failed)
    raise RuntimeError(f"Серия завершена частично. Ошибки: {SUMMARY_DIR / 'errors.csv'}")
print("Готово: результаты сохранены для всех трёх наборов.")


,dataset,model,seed,split,comparison_id,mae,rmse,n_observed,best_epoch,parameters,fit_seconds,output_dir,qlike
0,synthetic,persistence,0,val,36989cce73a0,0.096973,0.121423,7200,None,0,0.0,/home/loktar/Desktop/money_coursework/results/...,NaN
1,synthetic,persistence,0,test,36989cce73a0,0.096982,0.121153,14412,None,0,0.0,/home/loktar/Desktop/money_coursework/results/...,NaN
2,moex,persistence,0,val,362a12d77352,0.000167,0.000364,2371,None,0,0.0,/home/loktar/Desktop/money_coursework/results/...,-7.368911
3,moex,persistence,0,test,362a12d77352,0.000190,0.000805,2283,None,0,0.0,/home/loktar/Desktop/money_coursework/results/...,-7.147614
4,metr_la,persistence,0,val,c43da30bee45,2.631560,4.372454,660000,None,0,0.0,/home/loktar/Desktop/money_coursework/results/...,NaN
5,metr_la,persistence,0,test,c43da30bee45,2.812904,4.620502,1247252,None,0,0.0,/home/loktar/Desktop/money_coursework/results/...,NaN


Сопоставимые запуски: /home/loktar/Desktop/money_coursework/results/summaries/persistence/20261124T081851_533783Z/comparison.csv
Готово: результаты сохранены для всех трёх наборов.
